In [1]:
import sys
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.utils as vutils

In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

In [3]:
cwd = Path.cwd()
project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done")

Done


In [4]:
from Scripts.utils import load_mnist_dataset

In [5]:
data_path = project_root / "data"
train_dataloader, test_dataloader = load_mnist_dataset(
    data_path=data_path,
    batch_size=128
)

In [6]:
next(iter(train_dataloader))

[tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         [[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         [[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         ...,
 
 
         [[[-1., -1., -1.,  ..., -

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_size: int):
        super().__init__()

        self.tanh   =   nn.Tanh()
        self.relu   =   nn.ReLU()
        self.layer1 =   nn.Linear(in_features=noise_size, out_features=7*7*256)
        self.bn1    =   nn.BatchNorm2d(num_features=7*7*256)
        self.layer2 =   nn.ConvTranspose2d(in_channels=256, out_channels=256, kernel_size=3, stride=2)
        self.bn2    =   nn.BatchNorm2d(num_features=256)
        self.layer3 =   nn.ConvTranspose2d(in_channels=256, out_channels=256, kernel_size=3, stride=1)
        self.bn3    =   nn.BatchNorm2d(num_features=256)
        self.layer4 =   nn.ConvTranspose2d(in_channels=256, out_channels=256, kernel_size=3, stride=2)
        self.bn4    =   nn.BatchNorm2d(num_features=256)
        self.layer5 =   nn.ConvTranspose2d(in_channels=256, out_channels=256, kernel_size=3, stride=1)

    def forward(self, X):                       # (no_of_samples, noise_size)
        X = self.layer1(X)                      # (no_of_samples, 256*7*7)
        X = self.view(X.shape[0], 256, 7, 7)    # (no_of_samples, 256, 7, 7)
        X = self.bn1(X)                         # (no_of_samples, 256, 7, 7)
        X = self.relu(X)                        # (no_of_samples, 256, 7, 7)
        X = self.layer2(X)
        X = self.bn2(X)
        X = self.relu(X)
        X = self.layer3(X)
        X = self.bn3(X)
        X = self.relu(X)
        X = self.layer4(X)
        X = self.bn4(X)
        X = self.relu(X)
        X = self.layer5(X)
        X = self.tanh(X)

        return X